# 4 · Portfolio risk (NLP / QCP)

**Finance.** You have money to spread over 12 sector funds. Each fund has an expected return and a risk (volatility), and funds move together to different degrees. Markowitz's idea: **choose weights that balance return against risk**, where risk is the variance of the whole portfolio,

$$\text{risk}(w) = \sum_{s}\sum_{s'} w_s\, \Sigma_{s,s'}\, w_{s'}$$

That's **quadratic** in the weights, so the model is nonlinear (a QCP).

> The data are **simulated** daily returns for 12 fictional sector funds over three years (`data/sector_returns_simulated.csv`). They are realistic in shape, but they are not real market data.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gamspy as gp

returns = pd.read_csv("data/sector_returns_simulated.csv", index_col="date", parse_dates=True)
mu  = returns.mean() * 252          # expected yearly return
cov = returns.cov() * 252           # yearly covariance matrix
vol = np.sqrt(np.diag(cov))

fig = go.Figure(go.Scatter(x=vol, y=mu, text=mu.index, mode="markers+text",
                           textposition="top right", textfont_size=9, marker_color="#30383B"))
fig.update_layout(title="12 sector funds", xaxis_title="Risk (yearly volatility)",
                  yaxis_title="Expected yearly return", template="simple_white", width=700, height=450)
fig.show()

## The model

$$\min_w \; w^\top \Sigma w \; - \; \lambda\, \mu^\top w \qquad \text{s.t.} \quad \sum_s w_s = 1,\; w \ge 0$$

λ is **risk appetite**: λ = 0 means "least risk at any return"; a large λ chases return.

The one new GAMSPy idea is the `Alias`: the covariance needs two indices over the same set.

In [ ]:
m = gp.Container()
s  = gp.Set(m, "s", description="funds", records=mu.index.tolist())
s2 = gp.Alias(m, "s2", s)

MU  = gp.Parameter(m, "mu",  domain=s,      records=mu.reset_index())
COV = gp.Parameter(m, "cov", domain=[s, s2], records=cov.stack().reset_index())
lam = gp.Parameter(m, "lam", description="risk appetite", records=1.0)

w = gp.Variable(m, "w", domain=s, type="positive", description="weight of fund s")

budget = gp.Equation(m, "budget")
budget[...] = gp.Sum(s, w[s]) == 1

risk = gp.Sum((s, s2), w[s] * COV[s, s2] * w[s2])
ret  = gp.Sum(s, MU[s] * w[s])

portfolio = gp.Model(m, "portfolio", equations=[budget], problem="QCP",
                     sense="min", objective=risk - lam * ret)
portfolio.solve(solver="CPLEX")

weights = w.records.set_index("s")["level"]
print(f"return {mu @ weights:.1%}, risk {np.sqrt(weights @ cov @ weights):.1%}")
weights[weights > 0.005].round(3)

## The efficient frontier

Re-solve for a range of λ values. Each solve gives one point: the best return for that level of risk.

In [ ]:
def frontier(model, lam, w, lambdas):
    pts = []
    for L in lambdas:
        lam.setRecords(L)
        model.solve(solver="CPLEX")
        W = w.records.set_index("s")["level"].reindex(mu.index).fillna(0)
        pts.append({"lam": L, "return": mu @ W, "risk": np.sqrt(W @ cov @ W),
                    "funds": int((W > 0.005).sum())})
    return pd.DataFrame(pts)

lambdas = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1, 1.5, 2, 3]
front = frontier(portfolio, lam, w, lambdas)
front.round(3)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=vol, y=mu, text=mu.index, mode="markers", marker_color="#9AA3AE", name="single funds"))
fig.add_trace(go.Scatter(x=front["risk"], y=front["return"], mode="lines+markers",
                         line_color="#F4961A", name="efficient frontier"))
fig.update_layout(xaxis_title="Risk (yearly volatility)", yaxis_title="Expected yearly return",
                  template="simple_white", width=700, height=450)
fig.show()

**Your turn:** pick a risk level you'd be comfortable with (say 10% volatility). Which λ gets you there, and which funds are in your portfolio? Compare with your neighbor.

In [ ]:
lam.setRecords(0.5)      # try your own value
portfolio.solve(solver="CPLEX")
weights = w.records.set_index("s")["level"]
print(f"return {mu @ weights:.1%}, risk {np.sqrt(weights @ cov @ weights):.1%}")
top = weights[weights > 0.005].round(3).sort_values()
fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h", marker_color="#F4961A"))
fig.update_layout(title="Your portfolio", template="simple_white", width=700, height=400)
fig.show()

## A real-world rule: hold at most k funds

Managers often cap the number of holdings. That's a yes/no decision per fund, so we bring back the binaries from Act 2:

$$w_s \le y_s, \qquad \sum_s y_s \le k, \qquad y_s \in \{0, 1\}$$

Quadratic objective + binaries = **MIQCP**, a mixed-integer nonlinear model. CPLEX can solve it because the risk term is convex.

In [ ]:
y = gp.Variable(m, "y", domain=s, type="binary", description="1 if fund s is held")
k = gp.Parameter(m, "k", description="max number of funds", records=3)

link = gp.Equation(m, "link", domain=s)
max_funds = gp.Equation(m, "max_funds")
link[s] = w[s] <= y[s]
max_funds[...] = gp.Sum(s, y[s]) <= k

capped = gp.Model(m, "capped", equations=[budget, link, max_funds], problem="MIQCP",
                  sense="min", objective=risk - lam * ret)
front_k3 = frontier(capped, lam, w, lambdas)
front_k3.round(3)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=front["risk"], y=front["return"], mode="lines+markers",
                         line_color="#F4961A", name="any number of funds"))
fig.add_trace(go.Scatter(x=front_k3["risk"], y=front_k3["return"], mode="lines+markers",
                         marker_symbol="square", line_color="#30383B", name="at most 3 funds"))
fig.update_layout(xaxis_title="Risk (yearly volatility)", yaxis_title="Expected yearly return",
                  template="simple_white", width=700, height=450)
fig.show()

**Discuss:** where does the 3-fund frontier fall behind, and why does it match at high λ?

**Stretch**

1. Try k = 1, 2, 5. How many funds do you need before the cap stops mattering?
2. Replace the objective with "minimize risk subject to return ≥ 8%". Which formulation do you find easier to explain to a client?

### Convex vs. nonconvex

This model is **convex** (the covariance matrix makes the risk a bowl), so any local optimum is the global one, and CPLEX can prove it. The pooling problem in notebook 2 was **nonconvex**, and a local solver gave different answers from different starting points. Knowing which kind of model you have tells you whether to trust a single solve.